# PA2 stage 01: SFT (Task 3) and base-model baseline (Task 2)

Trains Qwen2.5-0.5B on 30k UltraChat examples on `cuda:0` with the from-scratch loop in
`scripts/train_sft.py`, while `cuda:1` generates base-model baselines. Outputs land in
`/kaggle/working/small-model-alignment-starter/outputs/`.

In [ ]:
REPO_URL = "https://github.com/usmanbinnaeem/small-model-alignment-starter.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git log -1 --format="%h %s"

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib
!python -m pip install -q -e . --no-deps

In [ ]:
import sys, torch, transformers, datasets
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB")
assert torch.cuda.device_count() == 2, "Expected GPU T4 x2"

In [ ]:
# Link the attached prepared-data dataset to data/prepared, wherever Kaggle mounted it.
import glob
hits = glob.glob("/kaggle/input/**/sft_train.jsonl", recursive=True)
assert hits, "attach the pa2-alignment-data dataset"
DATA_DIR = os.path.dirname(hits[0])
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/prepared"):
    os.symlink(DATA_DIR, "data/prepared")
!ls -la data/prepared/
!PYTHONPATH=. python -m pytest -q tests/

In [ ]:
# Resume support: copy this stage's run folders back from a previous run's attached output.
import shutil
for run in ['sft', 'gen']:
    local = os.path.join("outputs", run)
    if os.path.exists(local):
        continue
    found = sorted(glob.glob(f"/kaggle/input/**/small-model-alignment-starter/outputs/{run}", recursive=True))
    if found:
        print(f"restoring outputs/{run} from {found[-1]}")
        shutil.copytree(found[-1], local, symlinks=False)
    else:
        print(f"outputs/{run}: starting fresh")
!du -sh outputs/* 2>/dev/null || true

In [ ]:
# Task 2 baseline on the idle second GPU while SFT trains on the first.
# Two decoding settings, so the shared SFT/DPO setting can be chosen with evidence.
baseline_cmd = " && ".join(
    f"python -u scripts/generate.py --model Qwen/Qwen2.5-0.5B --generator qwen2.5-0.5b-base "
    f"--eval_set eval_sets/{evalset}.json --out outputs/gen/baseline/{tag}/{evalset}.json "
    f"--batch_size 32 --max_new_tokens 512 --repetition_penalty {rp}"
    for tag, rp in (("greedy", 1.0), ("greedy_rp1.1", 1.1))
    for evalset in ("alpaca_eval_300_seed42", "hh_rlhf_eval_300_seed42")
)
os.makedirs("outputs/gen/baseline", exist_ok=True)
baseline = subprocess.Popen(baseline_cmd, shell=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": "1", "PYTHONPATH": "."},
                            stdout=open("outputs/gen/baseline/generate.log", "w"), stderr=subprocess.STDOUT)
print("baseline generation started, pid", baseline.pid)

In [ ]:
!mkdir -p outputs/sft
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -u scripts/train_sft.py --config configs/sft.yaml 2>&1 | tee -a outputs/sft/train.log

In [ ]:
print("waiting for baseline generation ...")
print("exit code", baseline.wait())
!tail -n 5 outputs/gen/baseline/generate.log

In [ ]:
!PYTHONPATH=. python scripts/plot_metrics.py outputs/sft
!cat outputs/sft/summary.json

In [ ]:
# Task 6 sanity check: a few prompts through the SFT best checkpoint.
!PYTHONPATH=. python scripts/generate.py --model outputs/sft/best --generator sft-sanity \
    --eval_set eval_sets/alpaca_eval_300_seed42.json --out outputs/gen/sanity/sft_best.json --limit 5 --batch_size 5
import json
for r in json.load(open("outputs/gen/sanity/sft_best.json")):
    print("###", r["instruction"][:200], f"[{r['finish_reason']}]")
    print(r["output"][:800], "\n")

In [ ]:
# Small artifacts (no checkpoints) as one zip, per the assignment's saving guidance.
!cd outputs && zip -qr ../sft_run_outputs.zip sft gen -x "sft/best/*" "sft/final/*" && cd .. && mv sft_run_outputs.zip /kaggle/working/
!du -sh outputs/sft/best outputs/sft/final /kaggle/working/sft_run_outputs.zip